In [1]:
import os
import json
import shutil
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_openai import (
    OpenAIEmbeddings,
    OpenAI
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from pathlib import Path
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever
)

from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore, create_kv_docstore

load_dotenv(override=True)



True

In [2]:
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
EMBEDDING_MODEL_SMALL = "text-embedding-3-small"
CHROMA_PERSIST_DIR = str(PROJECT_ROOT / "data" / "chroma_db")
COLLECTION_NAME = "jd_intelligence_baseline_500d"

In [3]:
db_path = PROJECT_ROOT / "data" / "chroma_db"

if db_path.exists():
    if db_path.is_dir():
        shutil.rmtree(db_path)  
        print("Directory deleted successfully.")
    else:
        os.remove(db_path)     
else:
    print("Path does not exist.")

Path does not exist.


In [19]:
with open(PROJECT_ROOT / "checkpoint.json", 'r', encoding='utf-8') as f:
    checkpoint = json.load(f)

checkpoint

[{'ingested_raw_filenames': [], 'normalized_output_filenames': []}]

## **1. Document Loader and Data Normalization**

In [5]:
def load_and_normalize_jd(document_path: Path) -> str:

    loader = TextLoader(document_path)
    documents = loader.load()

    text_data = documents[0].page_content
    # Fixed Syntax
    data = [line for line in text_data.split("\n\n") if line != '\n']

    final_text = []

    for block in data:
        block = block.strip()
        if "\n" in block:
            # Split the sub-lines inside this block
            sub_lines = block.split("\n")
            # Join them back with a single SPACE, filtering out empty entries
            block = ' '.join([sl.strip() for sl in sub_lines if sl.strip()])

        final_text.append(block)

    return ' '.join(final_text)


In [ ]:


directory_path = PROJECT_ROOT / "data" / "raw_jds"
normalized_files = []
metadata = {}

for file in os.listdir(directory_path):
    # Track files using the clean string filename ('visa_de.txt')
    if file not in checkpoint[0]["ingested_raw_filenames"]:
        full_file_path = directory_path / file
        
        if full_file_path.is_file():
            normalized_files.append(file)
            print(f"Successfully processed: {file}")

            # Extract, clean, and write the text file
            cleaned_data = load_and_normalize_jd(full_file_path)

            if "Openings:" in cleaned_data:

                metadata["posted_date"] = (
                    cleaned_data.split("Posted:")[1]
                    .split("Openings:")[0]
                    .strip()
                )

                metadata["openings"] = (
                    cleaned_data.split("Openings:")[1]
                    .split("Applicants:")[0]
                    .strip()
                )

            else:

                metadata["posted_date"] = (
                    cleaned_data.split("Posted:")[1]
                    .split("Applicants:")[0]
                    .strip()
                )

                metadata["openings"] = None


            metadata["applicants"] = (
                cleaned_data.split("Applicants:")[1]
                .split("Save")[0]
                .strip()
            )
            
            output_path = PROJECT_ROOT / "data" / "cleaned_jds" / f"{full_file_path.stem}.txt"
            
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(cleaned_data)
            
            # Appending the text filename string so JSON can save it safely
            checkpoint[0]["ingested_raw_filenames"].append(file)

print(f"\nTotal new files processed normalized: {len(normalized_files)}")

with open(PROJECT_ROOT / "checkpoint.json", 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, indent=4)

print("Checkpoint database updated cleanly.")

Successfully processed: WissenTechnology_JavaDeveloper.txt
{'posted_date': '1 day ago', 'openings': None, 'applicants': '100+'}
 100+ 
Successfully processed: WissenTechnology_DataEngineer.txt
{'posted_date': '1 week ago', 'openings': None, 'applicants': '100+'}
 100+ 
Successfully processed: TechSMCSquaredGCC_DataEngineer.txt
{'posted_date': '2 days ago', 'openings': '1', 'applicants': '100+'}
 100+ 
Successfully processed: RavenrockDevelopments_DataEngineer.txt
{'posted_date': '3 days ago', 'openings': '1', 'applicants': '100+'}
 100+ 
Successfully processed: TheBusinessResearchCompany_SeniorDataEngineerDataAnalyticsEngineer.txt
{'posted_date': '1 week ago', 'openings': '1', 'applicants': '100+'}
 100+ 
Successfully processed: HimozoTalentConsultingPvtltd_SoftwareEngineer.txt
{'posted_date': '5 days ago', 'openings': '20', 'applicants': '100+'}
 100+ 
Successfully processed: FirstMeridianBusinessServices_AIDataEngineer.txt
{'posted_date': '2 days ago', 'openings': '10', 'applicants':

## **2. Text Splitter, Embeddings and Vector Store**

In [9]:
## Text Splitteing or Chunking
def chunk_jd_text(splitter:RecursiveCharacterTextSplitter, source_path:Path) ->  list[Document]:

    cleaned_text = ""

    with open(source_path, 'r', encoding="utf-8") as f:
        cleaned_text = f.read()

    chunks = splitter.create_documents(
        texts=[cleaned_text],
        metadatas=[
            {
                "source" : source_path.stem,
                "Description" : f"{(source_path.stem).split("_")[0]} {(source_path.stem).split("_")[1]} job description"
            }
        ]
    )
    
    return chunks
    
    

In [10]:

cleaned_file_path = Path(PROJECT_ROOT / "data" / "cleaned_jds")

## 1. Text Splitter configuration
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

## 2. Low-dimension Embedding configuration
embeddings_engine = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_SMALL,
    dimensions=500,  # Your target evaluation baseline length
    max_retries=3
)

## 3. Persistent Storage Handler connection
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_engine,
    persist_directory=CHROMA_PERSIST_DIR
)

files_processed_this_run = 0

all_documents = []

for file in os.listdir(cleaned_file_path):
    if not file.startswith('.') and file.endswith('.txt'):
        
        if file not in checkpoint[0]["normalized_output_filenames"]:
            cleaned_full_file_path = cleaned_file_path / file
            
            # Extract live documents into memory
            chunked_data = chunk_jd_text(splitter=splitter, source_path=cleaned_full_file_path)
            
            if chunked_data:
                # Core fix: add_documents pushes directly to DB and returns a list of chunk IDs
                chunk_ids = vector_store.add_documents(chunked_data)
                all_documents.extend(chunked_data)
                files_processed_this_run += 1

                print(f"==================================================")
                print(f"SUCCESS: {file}")
                print(f" -> Generated Chunks: {len(chunked_data)}")
                print(f" -> Indexed Chunk IDs: {len(chunk_ids)} registered keys")
                print(f" -> Sample ID range: {chunk_ids[0]} ... {chunk_ids[-1]}")
                print(f"==================================================\n")

            # Mark the file as processed in memory state
            checkpoint[0]['normalized_output_filenames'].append(file)

print(f"Processing Batch Complete! {files_processed_this_run} new files indexed safely.")

# 4. Commit tracking records to disk
with open(PROJECT_ROOT / "checkpoint.json", 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, indent=4)

print("Checkpoint ledger committed safely to checkpoint.json.")

/var/folders/gp/0r9x4vj97xn9cbjqk5n01ykc0000gn/T/ipykernel_2508/26625458.py:18: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


SUCCESS: capgmini_javadeveloper.txt
 -> Generated Chunks: 2
 -> Indexed Chunk IDs: 2 registered keys
 -> Sample ID range: 9ce11667-7a69-4f64-9035-6efea2248af5 ... 6bce2cff-fae7-41c8-89f8-d8f2ebffe555

SUCCESS: hcl_tech_java_developer.txt
 -> Generated Chunks: 6
 -> Indexed Chunk IDs: 6 registered keys
 -> Sample ID range: 9dc84b41-5418-437f-b269-74028be7f371 ... 0a66d871-c3cb-4fc6-a49e-b4e8cb1781b8

SUCCESS: glballogic_javadeveloper.txt
 -> Generated Chunks: 7
 -> Indexed Chunk IDs: 7 registered keys
 -> Sample ID range: cc43aa2d-c356-466d-866c-5cb30ac68ce5 ... 273aaa5c-88a4-45bf-91e9-b68b9605f91b

SUCCESS: tiger_analytics_de.txt
 -> Generated Chunks: 11
 -> Indexed Chunk IDs: 11 registered keys
 -> Sample ID range: d26bf8e5-48b8-4b08-8f08-57182dd5bcf0 ... 8f414b95-aa87-4596-a9aa-f27e7450d829

SUCCESS: innodata_india_de.txt
 -> Generated Chunks: 5
 -> Indexed Chunk IDs: 5 registered keys
 -> Sample ID range: 9131bfef-fc9a-4742-a5a2-85089f5dea83 ... e6c2fb73-5dc9-4c24-9894-55cd72718854


-- not required right now

In [11]:
## Retrival
## 1. Similarity Search
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2
    }
)

query = "Looking for a GCP data engineer Role without Spark or Databricks as a skill"

results_with_scores = vector_store.similarity_search_with_score(query)

print("--- Similarity Search Results with L2 Distance Scores ---")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n[Match {i}] Source File: {doc.metadata.get('source')}.txt")
    print(f"    Distance Score: {score:.4f}")  # Smaller = Closer/More accurate match
    print(f"    Content Snippet: {doc.page_content[:150]}...")

--- Similarity Search Results with L2 Distance Scores ---

[Match 0] Source File: innodata_india_de.txt
    Distance Score: 0.5373
    Content Snippet: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataf...

[Match 1] Source File: tcs_gcp_de.txt
    Distance Score: 0.6690
    Content Snippet: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Des...

[Match 2] Source File: tcs_gcp_de.txt
    Distance Score: 0.6846
    Content Snippet: in programming languages such as Python, PySpark data manipulation and engineering tasks. •Expertise in SQL and NoSQL databases •In-depth knowledge of...

[Match 3] Source File: innodata_india_de.txt
    Distance Score: 0.6872
    Content Snippet: and pipeline reliability. Skills Required : • Strong hands-on expertise with GCP services: BigQuery, Dataflow, Pub/Sub

In [12]:
## Similarity Search with Thershold

threshold = 0.5

retriever_thershould = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": threshold},
)

results = retriever_thershould.invoke(query)

print(f"=== Threshold = {threshold} — {len(results)} document(s) returned ===")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")


=== Threshold = 0.5 — 4 document(s) returned ===
  [1] metadata={'Description': 'innodata india job description', 'source': 'innodata_india_de'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
  [2] metadata={'source': 'tcs_gcp_de', 'Description': 'tcs gcp job description'}: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Desired Experience Range: 5-10 years Location: CHENNAI, PUNE, BENGALURU, Hyderabad Mode of Interview : Virtual Date of Interview

In [13]:
## MMR 

lambda_values = [1.0, 0.7, 0.5 ,0.0]

for lm in lambda_values:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    results = retriever.invoke(query)
    
    topics = [doc.metadata for doc in results]
    print(f"=== lambda_mult={lm} ===")
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] topic={doc.metadata}: {doc.page_content}")
    print()

=== lambda_mult=1.0 ===
  [1] topic={'source': 'innodata_india_de', 'Description': 'innodata india job description'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
  [2] topic={'Description': 'tcs gcp job description', 'source': 'tcs_gcp_de'}: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Desired Experience Range: 5-10 years Location: CHENNAI, PUNE, BENGALURU, Hyderabad Mode of Interview : Virtual Date of Interview : 14-05-26 Desired Competencie

In [14]:
bm_retriever = BM25Retriever.from_documents(
    documents=all_documents
)

bm_retriever.k = 3

bm_results = bm_retriever.invoke(
    query
)

bm_results

[Document(metadata={'source': 'innodata_india_de', 'Description': 'innodata india job description'}, page_content='to data pipelines for AI/ML (vector DB ingestion, embeddings, RAG pipelines, copilots, agents). • Familiarity with supply chain or data center operations data is a strong plus. Role: Data Engineer Industry Type: IT Services & Consulting Department: Engineering - Software & QA Employment Type: Full Time, Permanent Role Category: Software Development Education UG: B.Tech / B.E. in Any Specialization Key Skills Skills highlighted with ‘‘ are preferred keyskills ETL/ELT GCP Bigquery Cloud Storage'),
 Document(metadata={'source': 'jpmorganchase_leaddataengineer', 'Description': 'jpmorganchase leaddataengineer job description'}, page_content='17+, Spring, Boot), with strong system design and distributed systems knowledge. Extensive experience designing, building, and optimizing ETL/ELT pipelines at scale, including batch and real-time data processing. Strong proficiency in PySpa

In [15]:
chroma_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

ensemble_retriever = EnsembleRetriever(
    retrievers=[chroma_retriever, bm_retriever],
    weights=[0.8, 0.2]
)

ensemble_results = ensemble_retriever.invoke(
    query
)

for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] {doc.metadata}: {doc.page_content}")

  [1] {'source': 'innodata_india_de', 'Description': 'innodata india job description'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
  [2] {'source': 'tcs_gcp_de', 'Description': 'tcs gcp job description'}: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Desired Experience Range: 5-10 years Location: CHENNAI, PUNE, BENGALURU, Hyderabad Mode of Interview : Virtual Date of Interview : 14-05-26 Desired Competencies: (Technical/Behavioural Competency

In [16]:
llm = OpenAI(temperature=0)

compressor = LLMChainExtractor.from_llm(
    llm=llm
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(query)

lambda_values = [0.0]

for lm in lambda_values:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    results = retriever.invoke(query)
    
    topics = [doc.metadata for doc in results]
    print(f"=== lambda_mult={lm} ===")
    for i, doc in enumerate(results, 1):
        print(len(doc.page_content))
        print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")
    print()

print("=" * 50)

print(len(compressed_docs[0].page_content))
print(compressed_docs)

=== lambda_mult=0.0 ===
495
  [1] metadata={'Description': 'innodata india job description', 'source': 'innodata_india_de'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
497
  [2] metadata={'Description': 'accenture data job description', 'source': 'accenture_data_platform_de'}: Job description Project Role :Data Platform Engineer Project Role Description :Assists with the data platform blueprint and design, encompassing the relevant data platform components. Collaborates with the Integration Architects and Data Architects to ensure cohesive i

### **Parent Document Retriever**

In [60]:
## For child Chunks
vector_store_parent_child = Chroma(
    collection_name="split_parent",
    embedding_function=embeddings_engine,
    persist_directory=CHROMA_PERSIST_DIR
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 70
)

## For parent chunks
fs = LocalFileStore(root_path=Path(PROJECT_ROOT / "data" / "doc_store"))

persistent_docstore = create_kv_docstore(fs)

parent_retirever = ParentDocumentRetriever(
    vectorstore=vector_store_parent_child,
    docstore=persistent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)



cleaned_data = []

for file in os.listdir(cleaned_file_path):

    if os.path.exists(Path(PROJECT_ROOT / "data" / "cleaned_jds" / file)):
       
        with open(Path(PROJECT_ROOT / "data" / "cleaned_jds" / file), 'r', encoding='utf-8') as f:
            cleaned_data.extend([Document(
                page_content=f.read(),
                metadata={
                        "source" : Path(file).stem,
                        "Description" : Path(file).stem.replace("_", " ") + " job description"
                }
            )]
            )

    
parent_results = parent_retirever.add_documents(cleaned_data)

parent_keys = list(persistent_docstore.yield_keys())
total_parents = len(parent_keys)

print(f"Total Parent Chunks generated and stored: {total_parents}")

print()

# Query ChromaDB for the total number of items in this specific collection
total_children = vector_store_parent_child._collection.count()

print(f"Total Child Chunks generated and indexed: {total_children}")


Total Parent Chunks generated and stored: 43

Total Child Chunks generated and indexed: 158


In [68]:

parent_retriever = vector_store_parent_child.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.3},
)


parent_retrieve_results = parent_retriever.invoke(query)

print(parent_retrieve_results)

[Document(metadata={'Description': 'innodata india de job description', 'doc_id': '33f0e6fb-de80-4004-8088-d2e58c767b35', 'source': 'innodata_india_de'}, page_content='Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP,'), Document(metadata={'doc_id': '7ed7d163-e82f-478d-a3fb-042032840a2e', 'source': 'tcs_gcp_de', 'Description': 'tcs gcp de job description'}, page_content='Mode of Interview : Virtual Date of Interview : 14-05-26 Desired Competencies: (Technical/Behavioural Competency) Required Skills: •Expertise in Google Data Engineering services Big Query, Dataproc etc. •Strong experience in programming languages such as Python, PySpark data manipulation and'), Document(metadata={'doc_id': '4175fe82-28b2-4712-b57e-4b456bf4aa47', 'source': 'visa

In [69]:
topics = [doc.metadata for doc in parent_retrieve_results]
print(f"=== lambda_mult={0.3} ===")
for i, doc in enumerate(parent_retrieve_results, 1):
    print(len(doc.page_content))
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")

=== lambda_mult=0.3 ===
295
  [1] metadata={'Description': 'innodata india de job description', 'doc_id': '33f0e6fb-de80-4004-8088-d2e58c767b35', 'source': 'innodata_india_de'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP,
292
  [2] metadata={'doc_id': '7ed7d163-e82f-478d-a3fb-042032840a2e', 'source': 'tcs_gcp_de', 'Description': 'tcs gcp de job description'}: Mode of Interview : Virtual Date of Interview : 14-05-26 Desired Competencies: (Technical/Behavioural Competency) Required Skills: •Expertise in Google Data Engineering services Big Query, Dataproc etc. •Strong experience in programming languages such as Python, PySpark data manipulation and
284
  [3] metadata={'doc_id': '4175fe82-28b2-4712-b57e-4b456bf4aa47', 'source': 'visa_de', 'D

In [71]:
for i, doc in enumerate(parent_retirever.invoke(query), 1):
    print(len(doc.page_content))
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")

799
  [1] metadata={'source': 'innodata_india_de', 'Description': 'innodata india de job description'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend end-to-end data and BI solutions, spanning extraction, storage, transformation, and visualization layers. • Partner with supply chain, real estate, and AI/ML teams to provide pipelines for AI solutions (e.g., RAG ingestion, Copilot integration, multi-agent workflows). • Ensure data governance, lineage,
795
  [2] metadata={'source': 'innodata_india_de', 'Description': 'innodata india de job descrip